# RappiPlus Business Performance Analysis

End-to-end business analysis combining **Python, SQL, statistical testing, and Tableau** to evaluate data quality, profitability, sales behavior, conversion, retention, and a checkout A/B test.

**Tableau Public:** [RappiPlus Business Performance Story](https://public.tableau.com/views/RappiPlusBusinessPerformanceDashboard/RappiPlusBusinessPerformanceStory?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)


## Executive Summary

- **Adjusted Revenue:** $9.64M
- **Adjusted Product Cost:** $3.83M
- **Adjusted Product-Level Profit:** $5.78M
- **Average Order Value:** $385.97
- **Average Items per Order:** 1.50
- **Top-selling product:** Vacuum-Pro-Black — 6,284 units
- **Final funnel conversion:** 80.04%
- **Best week-3 retention:** February — 43.98%
- **Checkout A/B test:** no statistically significant conversion lift (p = 0.4161)

Extreme-quantity records account for **81.45% of reported revenue** and **91.12% of known product cost**, so raw and adjusted views are kept separate.


## 1. Data Loading and Quality Checks


In [ ]:
import pandas as pd

orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

print(orders.shape, catalog.shape, marketing.shape)
print(orders.isna().sum())
print('Order duplicates:', orders.duplicated().sum())


### Key cleaning decisions

- Missing marketing channels were recovered from campaign IDs when possible.
- Invalid quantities were flagged instead of silently dropped.
- Extreme positive quantities were identified with the IQR rule.
- 300 orders with unknown country were retained because their sales information was valid.
- Orders without enough product-cost information were retained for revenue analysis but not treated as zero-cost profit.


In [ ]:
# Flag invalid and extreme quantities
orders['cantidad_invalida'] = orders['cantidad'] <= 0

valid_qty = orders.loc[orders['cantidad'] > 0, 'cantidad']
q1 = valid_qty.quantile(0.25)
q3 = valid_qty.quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr
orders['cantidad_extrema'] = orders['cantidad'] > upper_limit

print('IQR upper limit:', upper_limit)
print('Extreme positive quantities:', orders['cantidad_extrema'].sum())
print('Invalid quantities:', orders['cantidad_invalida'].sum())


## 2. Profitability and Sales Performance

The notebook preserves both the raw financial view and an adjusted operating view. The adjusted view excludes extreme and invalid quantities so typical order behavior is not dominated by a few unusually large records.


In [ ]:
# Join product costs
orders_cost = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto', how='left'
)

reported_revenue = orders_cost['monto_total'].sum()
known_product_cost = (orders_cost['cantidad'] * orders_cost['costo_unitario']).sum()
marketing_spend = marketing['gasto'].sum()
reported_profit_after_marketing = reported_revenue - known_product_cost - marketing_spend

print('Reported Revenue:', round(reported_revenue, 2))
print('Known Product Cost:', round(known_product_cost, 2))
print('Marketing Spend:', round(marketing_spend, 2))
print('Estimated Profit After Marketing:', round(reported_profit_after_marketing, 2))


In [ ]:
# Adjusted sales view used in Tableau
adjusted = orders_cost[(~orders_cost['cantidad_extrema']) & (~orders_cost['cantidad_invalida'])].copy()

adjusted_revenue = adjusted['monto_total'].sum()
adjusted_cost = (adjusted['cantidad'] * adjusted['costo_unitario']).sum()
adjusted['profit'] = adjusted['monto_total'] - adjusted['cantidad'] * adjusted['costo_unitario']
adjusted_profit = adjusted['profit'].sum()

avg_order_value = adjusted['monto_total'].mean()
avg_items = adjusted['cantidad'].mean()
units_by_product = adjusted.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)

print('Adjusted Revenue:', round(adjusted_revenue, 2))
print('Adjusted Cost:', round(adjusted_cost, 2))
print('Adjusted Profit:', round(adjusted_profit, 2))
print('Average Order Value:', round(avg_order_value, 2))
print('Average Items per Order:', round(avg_items, 2))
print(units_by_product.head())


### Profitability Results

- Raw reported revenue: **$51.99M**
- Known product cost: **$43.12M**
- Marketing spend: **$2.87M**
- Estimated raw profit after product cost and marketing: **$5.99M**
- Adjusted revenue: **$9.64M**
- Adjusted product cost: **$3.83M**
- Adjusted product-level profit before marketing: **$5.78M**

Marketing is analyzed separately because it is aggregated by date, country, campaign, and channel rather than by individual order or product. Allocating it directly in Tableau could duplicate spend across orders.


## 3. Conversion Funnel — SQL


In [ ]:
# SQL connection is intentionally omitted from the public portfolio version.
# The query below was run against the TripleTen training PostgreSQL database.

funnel_query = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY CASE
    WHEN nombre_evento = 'first_visit' THEN 1
    WHEN nombre_evento = 'select_item' THEN 2
    WHEN nombre_evento = 'add_to_cart' THEN 3
    WHEN nombre_evento = 'begin_checkout' THEN 4
    WHEN nombre_evento = 'add_payment_info' THEN 5
    WHEN nombre_evento = 'purchase' THEN 6
END;
"""
print(funnel_query)


### Funnel Results

- `first_visit`: 7,796 users
- `purchase`: 6,240 users
- Final conversion: **80.04%**
- Largest relative drop: `begin_checkout → add_payment_info` (**86.71%** stage conversion)
- `select_item → add_to_cart` is slightly above 100%, indicating the event data is not strictly sequential for every user.


## 4. Cohort Retention — SQL


In [ ]:
cohort_query = """
SELECT
    DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS cohorte,
    COUNT(DISTINCT u.id_usuario) AS usuarios_iniciales,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN u.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_1,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN u.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_2,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN u.id_usuario END) * 100.0 / COUNT(DISTINCT u.id_usuario), 2) AS semana_3
FROM users AS u
LEFT JOIN user_activity AS a ON u.id_usuario = a.id_usuario
GROUP BY DATE_TRUNC('month', CAST(u.fecha_registro AS DATE))
ORDER BY cohorte;
"""
print(cohort_query)


### Retention Results

Weekly retention remains around **40%–44%** across cohorts. The **February cohort** records the strongest week-3 retention at **43.98%**.


## 5. A/B Test — Checkout UI


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
conversiones = experiment.groupby('variante')['convirtio'].sum()
usuarios = experiment.groupby('variante')['convirtio'].count()

stat, p_value = proportions_ztest(count=conversiones, nobs=usuarios)
print('Z-statistic:', round(stat, 4))
print('p-value:', round(p_value, 4))


### A/B Test Result

- Control conversion: **15.69%**
- Treatment conversion: **16.29%**
- Z-statistic: **-0.8133**
- p-value: **0.4161**

Because p > 0.05, the null hypothesis is not rejected. The experiment does not provide statistically significant evidence that the checkout UI change improved conversion.


## 6. Tableau Story

The Tableau deliverable combines two dashboards:

1. **RappiPlus Executive Overview** — adjusted Revenue, Cost, Profit, Average Order Value, Average Items per Order, monthly trends, YTD revenue, and category performance.
2. **RappiPlus Order & Product Detail** — units sold, product-level Revenue, Cost, and Profit with interactive filters.

[View the interactive RappiPlus Business Performance Story](https://public.tableau.com/views/RappiPlusBusinessPerformanceDashboard/RappiPlusBusinessPerformanceStory?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)


## Key Business Takeaways

1. Investigate extreme-quantity records separately from typical operating performance.
2. Focus funnel optimization on the checkout-to-payment step.
3. Do not treat the tested checkout UI change as a proven conversion improvement.
4. Monitor product profitability alongside unit sales because volume alone does not guarantee profit.
